# Credit Card Fraud Detection
**Goal:** Build a classifier to detect fraudulent transactions on a highly imbalanced dataset.  
**Dataset:** [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, precision_score, recall_score, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load Data

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
df.describe()

## 3. Exploratory Data Analysis

### 3.1 Class Imbalance

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print(f"Legit transactions : {class_counts[0]} ({class_pct[0]:.2f}%)")
print(f"Fraud transactions : {class_counts[1]} ({class_pct[1]:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Legit', 'Fraud'], class_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Count')
axes[1].pie(class_counts.values, labels=['Legit', 'Fraud'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Distribution (Proportion)')
plt.tight_layout()
plt.show()

**Observation:** The dataset is severely imbalanced — only 0.17% of transactions are fraudulent. A naive model predicting 'Legit' every time scores ~99.8% accuracy but catches zero fraud. This is why we use F1 Score and AUC-ROC as our primary metrics, and apply SMOTE to handle the imbalance.

### 3.2 Transaction Amount: Fraud vs Legit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Class'] == 0]['Amount'].plot(kind='hist', bins=50, ax=axes[0],
                                     color='steelblue', alpha=0.7, label='Legit')
df[df['Class'] == 1]['Amount'].plot(kind='hist', bins=50, ax=axes[0],
                                     color='tomato', alpha=0.7, label='Fraud')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount')
axes[0].set_xlim(0, 2500)
axes[0].legend()

df.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_title('Amount by Class')
axes[1].set_xlabel('Class (0=Legit, 1=Fraud)')
axes[1].set_ylabel('Amount')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(f"Average fraud amount : ${df[df['Class']==1]['Amount'].mean():.2f}")
print(f"Average legit amount : ${df[df['Class']==0]['Amount'].mean():.2f}")

### 3.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, linewidths=0.3, annot=False)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Scale Amount and Time (V1-V28 are already PCA-transformed)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]}')
print(f'Test size  : {X_test.shape[0]}')
print(f'Fraud in train : {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Fraud in test  : {y_test.sum()} ({y_test.mean()*100:.2f}%)')

### 4.1 Handle Class Imbalance with SMOTE
SMOTE generates synthetic fraud samples to balance the training set. **Important:** SMOTE is applied only on the training set to avoid data leakage into the test set.

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE - Train size: {X_train_sm.shape[0]}')
print(pd.Series(y_train_sm).value_counts())

## 5. Model Training

### 5.1 Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)

lr_pred  = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_pred, target_names=['Legit', 'Fraud']))
print(f'AUC-ROC: {roc_auc_score(y_test, lr_proba):.4f}')

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, rf_pred, target_names=['Legit', 'Fraud']))
print(f'AUC-ROC: {roc_auc_score(y_test, rf_proba):.4f}')

## 6. Evaluation

### 6.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pred, title in zip(axes,
                            [lr_pred, rf_pred],
                            ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 6.2 AUC-ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8, 6))
for proba, label, color in zip(
    [lr_proba, rf_proba],
    ['Logistic Regression', 'Random Forest'],
    ['steelblue', 'tomato']
):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {auc:.4f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('AUC-ROC Curve Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('results/roc_curve.png', dpi=150)
plt.show()

### 6.3 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

### 6.4 Model Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Precision (Fraud)': [precision_score(y_test, lr_pred), precision_score(y_test, rf_pred)],
    'Recall (Fraud)'   : [recall_score(y_test, lr_pred),    recall_score(y_test, rf_pred)],
    'F1 Score (Fraud)' : [f1_score(y_test, lr_pred),        f1_score(y_test, rf_pred)],
    'AUC-ROC'          : [roc_auc_score(y_test, lr_proba),  roc_auc_score(y_test, rf_proba)]
})
results.set_index('Model', inplace=True)
results.round(4)

## 7. Conclusion

**Logistic Regression** serves as a strong interpretable baseline but is limited by its assumption of linear decision boundaries — fraud patterns in this dataset are non-linear, which hurts its recall on the fraud class.

**Random Forest** outperforms Logistic Regression across all key metrics (F1 Score, AUC-ROC, Recall on fraud). Its ability to model non-linear relationships and its robustness on imbalanced data make it the better choice for a production fraud detection system.

**Key takeaway:** In fraud detection, **Recall matters most** — missing a fraud is far more costly than a false alarm. Random Forest achieves higher recall on the fraud class, making it the preferred model here.